In [16]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

print("All imports successful!")

All imports successful!


LangChain is a framework for building applications with language models (LLMs). 
- It simplifies the process of integrating LLMs with external data sources (like documents, databases, or APIs) 
- It allows you to create intelligent apps such as chatbots, retrieval-augmented generation (RAG) systems, and more.

LangChain acts as the "glue" that connects:

* Your documents (TextLoader)---------------------------------------> Loads plain text documents into LangChain.
* Your text processing (TextSplitter)-------------------------------> Splits long text into manageable chunks.
* Your embeddings (HuggingFaceEmbeddings)---------------------------> Generates embeddings for document chunks.
* Your retrieval system (FAISS)-------------------------------------> Stores and retrieves embeddings efficiently using similarity    search.It helps you find the most similar text chunks from your documents when someone asks a question.
* Your LLM (HuggingFacePipeline)------------------------------------> Wraps a Hugging Face model as an LLM.
* And your question-answering logic (RetrievalQA)-------------------> Connects retrieval and LLM to answer questions based on documents.

In [22]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Step 2: Load the document. A text loader is used to load the document as raw text into memory, so it can be processed further.

loader = TextLoader("llm_notes.txt", encoding="utf-8")
documents = loader.load()

# Step 3: Split the document. Each chunk has 300 characters, and 50 characters overlap with the next chunk for context continuity.

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = text_splitter.split_documents(documents[:5])  # limit size to reduce memory

print(f"Number of chunks: {len(docs)}")

# Step 4: Embed using Hugging Face sentence transformer

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# --------------------------------------------------
# Store embeddings in FAISS
# --------------------------------------------------

db = FAISS.from_documents(
    docs,
    embeddings
)

# --------------------------------------------------
# Step 5: Load FLAN-T5 directly
# --------------------------------------------------

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

model.eval()


# --------------------------------------------------
# Step 6: Create LangChain LLM wrapper
# --------------------------------------------------
from langchain_core.language_models.llms import LLM
from typing import Optional, List
from pydantic import Field

class FlanT5LLM(LLM):
    model: object = Field(exclude=True)
    tokenizer: object = Field(exclude=True)
    max_new_tokens: int = 256

    @property
    def _llm_type(self) -> str:
        return "flan-t5"

    def _call(
            self,
            prompt: str,
            stop: Optional[List[str]] = None,
            run_manager=None,
            **kwargs
    ) -> str:
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False
        )

        response = self.tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        return response


llm = FlanT5LLM(
    model=model,
    tokenizer=tokenizer
)

# Step 6: Build the Retrieval QA chain. First retrieves top 3 relevant text chunks from FAISS, then passes them to the LLM to answer your query.
#It enhances the LLM's ability to answer questions by grounding it in specific documents.

retriever = db.as_retriever(search_kwargs={"k": 3})
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

# Step 7: Ask a question

query = "What is tokenization in LLMs and why is it important?"
result = qa_chain(query)

# Step 8: Show results

print("\n Answer:")
print(result["result"])

print("\n Source Documents:")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n--- Source {i} ---")
    print(doc.page_content)


Number of chunks: 36


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
/var/folders/mk/3k8t72sj69l04zsskj14q1j00000gn/T/ipykernel_5857/649936550.py:108: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa_chain(query)



 Answer:
Tokenization is the process of converting text into smaller units called tokens.

 Source Documents:

--- Source 1 ---
RAG Example
Suppose a user asks: "What is tokenization in LLMs and why is it important?"

--- Source 2 ---
Tokenization
Tokenization is the process of converting text into smaller units called tokens. These tokens can represent words, parts of words, characters, or symbols depending on the tokenizer being used.

--- Source 3 ---
Tokenization is important because neural language models do not directly process raw text. They process numerical representations of tokens. Each token is mapped to a numerical token ID before being passed to the model.


In [23]:
# ==========================================
# STEP 1: LOAD DOCUMENT
# ==========================================

loader = TextLoader("llm_notes.txt", encoding="utf-8")
documents = loader.load()

print("Documents loaded:", len(documents))
print(documents[0].page_content[:500])


# ==========================================
# STEP 2: SPLIT DOCUMENT
# ==========================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Number of chunks:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\nChunk {i}:")
    print(doc.page_content)


# ==========================================
# STEP 3: CREATE EMBEDDINGS
# ==========================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ==========================================
# STEP 4: CREATE VECTOR DATABASE
# ==========================================

db = FAISS.from_documents(
    docs,
    embeddings
)

print("FAISS database created")


# ==========================================
# STEP 5: TEST RETRIEVAL
# ==========================================

retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

query = "What is tokenization in LLMs?"

retrieved_docs = retriever.invoke(query)

print("\nRetrieved documents:")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved {i} ---")
    print(doc.page_content)


# ==========================================
# STEP 6: LOAD FLAN-T5
# ==========================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.eval()

print("FLAN-T5 loaded")

Documents loaded: 1
Large Language Models (LLMs)
A Large Language Model (LLM) is a type of artificial intelligence model designed to understand and generate human language. LLMs are trained on large collections of text and learn patterns, relationships, and structures in language. Examples of LLMs include GPT, Llama, Claude, and Google's T5 and FLAN-T5 models.
LLMs can perform many language-related tasks, including text generation, question answering, summarization, translation, classification, and information extr
Number of chunks: 36

Chunk 0:
Large Language Models (LLMs)

Chunk 1:
A Large Language Model (LLM) is a type of artificial intelligence model designed to understand and generate human language. LLMs are trained on large collections of text and learn patterns, relationships, and structures in language. Examples of LLMs include GPT, Llama, Claude, and Google's T5 and

Chunk 2:
include GPT, Llama, Claude, and Google's T5 and FLAN-T5 models.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded
FAISS database created

Retrieved documents:

--- Retrieved 1 ---
RAG Example
Suppose a user asks: "What is tokenization in LLMs and why is it important?"

--- Retrieved 2 ---
Tokenization
Tokenization is the process of converting text into smaller units called tokens. These tokens can represent words, parts of words, characters, or symbols depending on the tokenizer being used.

--- Retrieved 3 ---
Tokenization is important because neural language models do not directly process raw text. They process numerical representations of tokens. Each token is mapped to a numerical token ID before being passed to the model.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 loaded
